In [2]:
import matplotlib.pyplot as plt
import pandas as pd

import numpy as np
import json

from google.colab import userdata
import os

In [3]:
%%capture
!pip install sacrebleu evaluate
!pip install -U huggingface_hub

In [4]:
from huggingface_hub import login
hf_token = userdata.get('hf_token')
login(token=hf_token)

In [5]:
!hf auth whoami

user:  dotnw


# Proof of concept

## Flores datasets

flores + grambank :
1) nucl1417 - Igbo - ibo
2) basq1248 - Basque - eus
3) kat	Geor	nucl1302	Georgian
4) ayr	Latn	cent2142	Central Aymara
5)

In [ ]:
%%capture
!pip install datasets

In [ ]:

from datasets import load_dataset
dataset = load_dataset("openlanguagedata/flores_plus", split="devtest")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/224 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/219 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/224 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/219 [00:00<?, ?it/s]

In [ ]:
basque_data = dataset.filter(lambda x: x['iso_639_3'] == 'eus' and x['iso_15924'] == 'Latn')
igbo_data = dataset.filter(lambda x: x['iso_639_3'] == 'ibo' and x['iso_15924'] == 'Latn')
geor_data = dataset.filter(lambda x: x['iso_639_3'] == 'kat' and x['iso_15924'] == 'Geor')
ayr_data = dataset.filter(lambda x: x['iso_639_3'] == 'ayr' and x['iso_15924'] == 'Latn')
english_data = dataset.filter(lambda x: x['iso_639_3'] == 'eng' and x['iso_15924'] == 'Latn')

igbo_data = igbo_data.sort("id")
basque_data = basque_data.sort("id")
geor_data = geor_data.sort("id")
ayr_data = ayr_data.sort("id")
english_data = english_data.sort("id")

igbo_sentences = igbo_data['text']
basque_sentences = basque_data['text']
geor_sentences = geor_data['text']
ayr_sentences = ayr_data['text']
english_sentences = english_data['text']


In [ ]:
# Verification
print(f"Igbo sentence 1: {igbo_sentences[0]}")
print(f"Basque sentence 1: {basque_sentences[0]}")
print(f"Georgian sentence 1: {geor_sentences[0]}")
print(f"Aymara sentence 1: {ayr_sentences[0]}")
print(f"English sentence 1: {english_sentences[0]}")

Igbo sentence 1: Ugbu a, anyị nwere ụmụ oke ọnwa anọ na-arịabu ọrịa shuga mana ha anaghị arịazi ya.
Basque sentence 1: “Diabetiko izateari utzi dioten 4 hilabeteko saguak ditugu orain” gehitu zuen.
Georgian sentence 1: „ჩვენ ახლა გვყავს 4 თვის ასაკის თაგვები, რომლებსაც დიაბეტი ჰქონდათ და ახლა აღარ აქვთ,“ — დასძინა მან.
Aymara sentence 1: “Jichhax 4 phaxsin achakunakanïtanwa kawknïriti janiw tiyawitiküpkti ukat tiyawitikupxirïnwa”, saw jupax artxatt’awayi.
English sentence 1: "We now have 4-month-old mice that are non-diabetic that used to be diabetic," he added.


## TYP data


In [6]:
para_df = pd.read_csv('/content/parameters.csv')

In [7]:
para_df.head()

,ID,Name,Description,ColumnSpec,Patrons,Grambank_ID_desc,Boundness,Flexivity,Gender_or_Noun_Class,Locus_of_Marking,Word_Order,Informativity
0,GB020,Are there definite or specific articles?,## Are there definite or specific articles?\r\...,NaN,JLA JC,GB020 ARTDef,NaN,NaN,NaN,NaN,NaN,definitearticles
1,GB021,Do indefinite nominals commonly have indefinit...,## Do indefinite/non-specific nominals commonl...,NaN,JLA JC,GB021 ARTIndef,NaN,NaN,NaN,NaN,NaN,indef
2,GB022,Are there prenominal articles?,## Are there prenominal articles?\r\n\r\n## Su...,NaN,JLA JC,GB022 ARTPre,NaN,NaN,NaN,NaN,0.0,NaN
3,GB023,Are there postnominal articles?,## Are there postnominal articles?\r\n\r\n## S...,NaN,JLA JC,GB023 ARTPost,NaN,NaN,NaN,NaN,1.0,NaN
4,GB024,What is the order of numeral and noun in the NP?,## What is the order of numeral and noun in th...,NaN,HJH,GB024 NUMOrder,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
para_df['Description'] = para_df['Description'].str.split('## Examples').str[0]

In [9]:
print(para_df['Description'][1])

## Do indefinite/non-specific nominals commonly have indefinite/non-specific articles?

## Summary
For this feature, we need explicit support that indefinite/non-specific NPs commonly occur with an indefinite/non-specific article. Three or more examples of indefinite/non-specific nominals occurring with an indefinite/non-specific article is sufficient to code 1. 

An indefinite article is a marker that accompanies the indefinite noun and expresses notions such as non-specificity and indefiniteness. Sometimes these notions of non-specificity and indefiniteness are summed up in the term ‘identifiability’. A language does not necessarily have both a non-specific/indefinite and a specific/definite marker. These markers often stem from very different sources (numerals and demonstratives) and need not be similar in formal expression or position. 

The formal expression is irrelevant; articles can be free, bound, or marked by suprasegmental markers such as tone. Articles are different from de

In [10]:
!unzip '/content/values.zip'

Archive:  /content/values.zip
  inflating: values.csv              


In [11]:
val_df = pd.read_csv('/content/values.csv')

In [12]:
val_df.head()

,ID,Language_ID,Parameter_ID,Value,Code_ID,Comment,Source,Source_comment,Coders
0,GB020-abad1241,abad1241,GB020,?,NaN,Author states there is a possible example of a...,s_OaPaul_Gabadi[17],Oa & Paul 2013:17,JLA
1,GB021-abad1241,abad1241,GB021,?,NaN,Author states there is a possible example of a...,s_OaPaul_Gabadi[17],Oa & Paul 2013:17,JLA
2,GB022-abad1241,abad1241,GB022,?,NaN,Author states there is a possible example of a...,s_OaPaul_Gabadi[17],Oa & Paul 2013:17,JLA
3,GB023-abad1241,abad1241,GB023,?,NaN,Author states there is a possible example of a...,s_OaPaul_Gabadi[17],Oa & Paul 2013:17,JLA
4,GB024-abad1241,abad1241,GB024,2,GB024-2,NaN,s_OaPaul_Gabadi[15],Oa & Paul 2013:15,JLA


In [13]:
subset_ibo = val_df[val_df['Language_ID'] == 'nucl1417']
subset_ibo.head()

,ID,Language_ID,Parameter_ID,Value,Code_ID,Comment,Source,Source_comment,Coders
284818,GB020-nucl1417,nucl1417,GB020,0,GB020-0,NaN,g_Emenanjo_Igbo_2015[279-300],Emenanjo (2015:279-300),JLA
284819,GB021-nucl1417,nucl1417,GB021,0,GB021-0,NaN,g_Emenanjo_Igbo_2015[279-300],Emenanjo (2015:279-300),JLA
284820,GB022-nucl1417,nucl1417,GB022,0,GB022-0,NaN,g_Emenanjo_Igbo_2015[279-300],Emenanjo (2015:279-300),JLA
284821,GB023-nucl1417,nucl1417,GB023,0,GB023-0,NaN,g_Emenanjo_Igbo_2015[279-300],Emenanjo (2015:279-300),JLA
284822,GB024-nucl1417,nucl1417,GB024,2,GB024-2,NaN,g_Emenanjo_Igbo_2015[279-282],Emenanjo (2015:279-282),JLA


In [14]:
subset_ibo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 194 entries, 284818 to 285011
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   ID              194 non-null    object
 1   Language_ID     194 non-null    object
 2   Parameter_ID    194 non-null    object
 3   Value           194 non-null    object
 4   Code_ID         188 non-null    object
 5   Comment         0 non-null      object
 6   Source          194 non-null    object
 7   Source_comment  194 non-null    object
 8   Coders          194 non-null    object
dtypes: object(9)
memory usage: 19.2+ KB


In [15]:
subset_basque = val_df[val_df['Language_ID'] == 'basq1248']
subset_basque.head()

,ID,Language_ID,Parameter_ID,Value,Code_ID,Comment,Source,Source_comment,Coders
38204,GB020-basq1248,basq1248,GB020,1,GB020-1,NaN,g_HualdeUrbana_Basque[119],Hualde (2003:119),JG;JLA;RHA
38205,GB021-basq1248,basq1248,GB021,1,GB021-1,NaN,g_HualdeUrbana_Basque[122],Hualde (2003:122),JG;JLA;RHA
38206,GB022-basq1248,basq1248,GB022,0,GB022-0,NaN,"g_HualdeUrbana_Basque[119, 122]","Hualde (2003:119, 122)",JG;JLA;RHA
38207,GB023-basq1248,basq1248,GB023,1,GB023-1,NaN,"g_HualdeUrbana_Basque[119, 122]","Hualde (2003:119, 122)",JG;JLA;RHA
38208,GB024-basq1248,basq1248,GB024,1,GB024-1,NaN,"g_HualdeUrbana_Basque[113, 118]","Hualde (2003:113, 118)",JG;JLA;RHA


In [16]:
subset_geor = val_df[val_df['Language_ID'] == 'nucl1302']
subset_geor.head()

,ID,Language_ID,Parameter_ID,Value,Code_ID,Comment,Source,Source_comment,Coders
283897,GB020-nucl1302,nucl1302,GB020,0,GB020-0,NaN,g_Hewitt_Georgian[62],Hewitt 1995:62,JSA;TWE
283898,GB021-nucl1302,nucl1302,GB021,0,GB021-0,NaN,g_Hewitt_Georgian[62],Hewitt 1995:62,JSA;TWE
283899,GB022-nucl1302,nucl1302,GB022,0,GB022-0,NaN,g_Hewitt_Georgian[62],Hewitt 1995:62,JSA;TWE
283900,GB023-nucl1302,nucl1302,GB023,0,GB023-0,NaN,g_Hewitt_Georgian[62],Hewitt 1995:62,JSA;TWE
283901,GB024-nucl1302,nucl1302,GB024,1,GB024-1,NaN,g_Hewitt_Georgian[54-56],Hewitt 1995:54-56,JSA;TWE


In [17]:
subset_cha = val_df[val_df['Language_ID'] == 'cham1312']
subset_cha.head()

,ID,Language_ID,Parameter_ID,Value,Code_ID,Comment,Source,Source_comment,Coders
73226,GB020-cham1312,cham1312,GB020,1,GB020-1,NaN,typ_Cooreman_Chamorro_1987[26],Cooreman 1987:26,NP
73227,GB021-cham1312,cham1312,GB021,0,GB021-0,Chung says that indefinite articles are common...,typ_Chung_Chamorro_Lexical[21],Chung 2012:21,NP
73228,GB022-cham1312,cham1312,GB022,1,GB022-1,NaN,typ_Cooreman_Chamorro_1987[26],Cooreman 1987:26,NP
73229,GB023-cham1312,cham1312,GB023,0,GB023-0,"Not mentioned, judged absent",g_Topping_Chamorro;typ_Chung_Chamorro_Agreement,Topping 1973; Chung 1998,NP
73230,GB024-cham1312,cham1312,GB024,1,GB024-1,NaN,typ_Chung_Chamorro_Agreement[47-48],Chung 1998:47-48,NP


In [18]:
subset_kor = val_df[val_df['Language_ID'] == 'kore1280']
subset_kor.head()

,ID,Language_ID,Parameter_ID,Value,Code_ID,Comment,Source,Source_comment,Coders
181676,GB020-kore1280,kore1280,GB020,0,GB020-0,NaN,g_Sohn_Korean[276-278],Sohn (1994:276-278),MM;NH
181677,GB021-kore1280,kore1280,GB021,0,GB021-0,Nominals that are unmarked for definiteness (b...,"g_Sohn_Korean[113-114, 224, 276-280];g_Sohn_Ko...","Sohn (1994: 113-114, 224, 276-280); Sohn (1999...",JP;MM;NH
181678,GB022-kore1280,kore1280,GB022,0,GB022-0,NaN,g_Sohn_Korean_1999[265],Sohn (1999:265),MM;NH
181679,GB023-kore1280,kore1280,GB023,0,GB023-0,NaN,"g_Sohn_Korean[113-114, 224, 276-280];g_Sohn_Ko...","Sohn (1994: 113-114, 224, 276-280); Sohn (1999...",JP;MM;NH
181680,GB024-kore1280,kore1280,GB024,2,GB024-2,NaN,"g_Sohn_Korean_1999[265, 296]","Sohn (1999:265, 296)",MM;NH


In [19]:
subset_eng = val_df[val_df['Language_ID'] == 'stan1293']
subset_eng.head()

,ID,Language_ID,Parameter_ID,Value,Code_ID,Comment,Source,Source_comment,Coders
357551,GB020-stan1293,stan1293,GB020,1,GB020-1,The definite article is 'the.',g_HuddlestonPullum_English[368],Huddleston & Pullum (2002:368),IC;JG;JLA
357552,GB021-stan1293,stan1293,GB021,1,GB021-1,The indefinite article is 'a' or at times 'an',g_HuddlestonPullum_English[371],Huddleston & Pullum (2002:371),IC;JG;JLA
357553,GB022-stan1293,stan1293,GB022,1,GB022-1,NaN,g_HuddlestonPullum_English[368],Huddleston & Pullum (2002:368),IC;JG;JLA
357554,GB023-stan1293,stan1293,GB023,0,GB023-0,NaN,g_HuddlestonPullum_English[368-371],Huddleston & Pullum (2002:368-371),IC;JG;JLA
357555,GB024-stan1293,stan1293,GB024,1,GB024-1,NaN,"g_HuddlestonPullum_English[363, 385-387]","Huddleston & Pullum (2002:363, 385-387)",IC;JG;JLA


In [20]:
subset_eng.info()

<class 'pandas.core.frame.DataFrame'>
Index: 195 entries, 357551 to 357745
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   ID              195 non-null    object
 1   Language_ID     195 non-null    object
 2   Parameter_ID    195 non-null    object
 3   Value           195 non-null    object
 4   Code_ID         195 non-null    object
 5   Comment         56 non-null     object
 6   Source          195 non-null    object
 7   Source_comment  195 non-null    object
 8   Coders          195 non-null    object
dtypes: object(9)
memory usage: 15.2+ KB


In [21]:
master_df = subset_ibo[['Parameter_ID', 'Value']].rename(columns={'Value': 'Igbo'})

In [22]:
master_df = master_df.merge(
    subset_basque[['Parameter_ID', 'Value']].rename(columns={'Value': 'Basque'}),
    on='Parameter_ID',
    how='left'
)
master_df = master_df.merge(
    subset_geor[['Parameter_ID', 'Value']].rename(columns={'Value': 'Georgian'}),
    on='Parameter_ID',
    how='left'
)
master_df = master_df.merge(
    subset_cha[['Parameter_ID', 'Value']].rename(columns={'Value': 'Chamorro'}),
    on='Parameter_ID',
    how='left'
)
master_df = master_df.merge(
    subset_kor[['Parameter_ID', 'Value']].rename(columns={'Value': 'Korean'}),
    on='Parameter_ID',
    how='left'
)
master_df = master_df.merge(
    subset_eng[['Parameter_ID', 'Value']].rename(columns={'Value': 'English'}),
    on='Parameter_ID',
    how='left'
)

In [23]:
master_df = master_df.merge(
    para_df[['ID', 'Name', 'Description']].rename(columns={'ID': 'Parameter_ID'}),
    on='Parameter_ID',
    how='left'
)

In [24]:
master_df.head()

,Parameter_ID,Igbo,Basque,Georgian,Chamorro,Korean,English,Name,Description
0,GB020,0,1,0,1,0,1,Are there definite or specific articles?,## Are there definite or specific articles?\r\...
1,GB021,0,1,0,0,0,1,Do indefinite nominals commonly have indefinit...,## Do indefinite/non-specific nominals commonl...
2,GB022,0,0,0,1,0,1,Are there prenominal articles?,## Are there prenominal articles?\r\n\r\n## Su...
3,GB023,0,1,0,0,0,0,Are there postnominal articles?,## Are there postnominal articles?\r\n\r\n## S...
4,GB024,2,1,1,1,2,1,What is the order of numeral and noun in the NP?,## What is the order of numeral and noun in th...


In [25]:
hyperlink_pattern = r'\[([^\]]+)\]\([^\)]+\)'
master_df['Description'] = master_df['Description'].str.replace(hyperlink_pattern, r'\1', regex=True)

In [26]:
def create_typ_prompt(df):
    all_columns = df.columns.tolist()
    start_idx = all_columns.index('Parameter_ID') + 1
    end_idx = all_columns.index('Name')
    languages = all_columns[start_idx:end_idx]

    all_prompts = {}
    for lang in languages:
        if lang == 'English':
            continue

        lang_prompts = []
        for _, row in df.iterrows():
            prompt = f"""
Feature ID: {row['Parameter_ID']}: {row['Name']}
{lang} Value: Code {row[lang]}

English Value: Code {row['English']}

—
Below is a short summary of the grammatical feature, an explanation of the process for assigning the feature’s
code, and examples of the feature from other languages including interlinear glossed text.
—
{row['Description']}
This is the end of the summary for feature {row['Parameter_ID']}: "{row['Name']}".
"""
            lang_prompts.append(prompt)

        all_prompts[lang] = lang_prompts

    return all_prompts, languages
prompts, languages = create_typ_prompt(master_df)


In [27]:
print(prompts['Igbo'][0])


Feature ID: GB020: Are there definite or specific articles?
Igbo Value: Code 0

English Value: Code 1

—
Below is a short summary of the grammatical feature, an explanation of the process for assigning the feature’s
code, and examples of the feature from other languages including interlinear glossed text.
—
## Are there definite or specific articles?

## Summary
An article is a marker that accompanies the noun and expresses notions such as (non-)specificity and (in)definiteness. Sometimes these notions of specificity and definiteness are summed up in the term 'identifiability'. The formal expression is irrelevant; articles can be free, bound, or marked by suprasegmental markers such as tone.
Articles are different from demonstratives in that demonstratives occur in a paradigm of markers that have a clear spatial deictic function. As demonstratives can grammaticalize into definite or specific articles, they form a natural continuum, making it hard to define discrete categories, but to 

In [28]:
final_prompts_typ = {}

separator = "\n___\n"

for lang, prompts_list in prompts.items():
    current_intro = f"""The following typological features describe the grammatical features of {lang} and English including word order, verbal tense, nominal case, and other language universals. Each feature is assigned a value that indicates the extent to which the language tends to exhibit that feature.
"""
    current_outro = f"—\nThis is the end of the typological feature summary for {lang} and English."

    body = separator.join(prompts_list)

    final_combined_prompt = f"{current_intro}\n{body}\n\n{current_outro}"

    final_prompts_typ[lang] = final_combined_prompt

print(final_prompts_typ['Chamorro'][:1000])

The following typological features describe the grammatical features of Chamorro and English including word order, verbal tense, nominal case, and other language universals. Each feature is assigned a value that indicates the extent to which the language tends to exhibit that feature.


Feature ID: GB020: Are there definite or specific articles?
Chamorro Value: Code 1

English Value: Code 1

—
Below is a short summary of the grammatical feature, an explanation of the process for assigning the feature’s
code, and examples of the feature from other languages including interlinear glossed text.
—
## Are there definite or specific articles?

## Summary
An article is a marker that accompanies the noun and expresses notions such as (non-)specificity and (in)definiteness. Sometimes these notions of specificity and definiteness are summed up in the term 'identifiability'. The formal expression is irrelevant; articles can be free, bound, or marked by suprasegmental markers such as tone.
Art


##

In [82]:
def generate_zeroshot_prompt(lang_name,sentence):
    instruction = f"""Translate the following sentence from {lang_name} to English: {sentence}

Now write the translation. If you are not sure what the translation should be, then give your best guess. Do not say that  you do not speak {lang_name}. Do not say you do not have enough information, you must make a guess. If your translation is wrong, that is fine, but you have to provide a translation.
Your translation must be on the first line of your response, with no other text before the translation. Only explain your reasoning after providing the translation.
It is crucial that you only give the translation on the first line of your response, otherwise you will fail.

Now write the translation:
{lang_name}: {sentence}
English:"""
    return instruction

In [ ]:
sentences = {
    'Igbo': igbo_data['text'],
    'Basque': basque_data['text'],
    'Georgian': geor_data['text'],
    'Aymar': ayr_data['text']
}

In [ ]:
all_zeroshot_prompts = {}
for lang_name, sents in sentences.items():
    all_zeroshot_prompts[lang_name] = [
        generate_zeroshot_prompt(lang_name, sent)
        for sent in sents
    ]

print(all_zeroshot_prompts['Igbo'][1])
print(all_zeroshot_prompts['Basque'][0])

Igbo is a language that exhibits specific typological characteristics. Translate the following sentence from Igbo to English: Dr. Ehud Ur, ọkammụta nke ọgwụ na Mahadum Dalhousie nọ na Halifax, Nova Scotia na onye isi oche nke ngalaba ụlọ ọgwụ na sayensi ahụ nke Òtù Ọrịa Shuga ndị Kanada dọrọ aka na ntị na nnyocha ahụ ka nọ n’oge ndị mbụ ya.

Now write the translation. If you are not sure what the translation should be, then give your best guess. Do not say that  you do not speak Igbo. Do not say you do not have enough information, you must make a guess. If your translation is wrong, that is fine, but you have to provide a translation.
Your translation must be on the first line of your response, with no other text before the translation. Only explain your reasoning after providing the translation.
It is crucial that you only give the translation on the first line of your response, otherwise you will fail.

Now write the translation:
Igbo: Dr. Ehud Ur, ọkammụta nke ọgwụ na Mahadum Dalh

In [ ]:
print(all_zeroshot_prompts['Aymar'][0])

Aymar is a language that exhibits specific typological characteristics. Translate the following sentence from Aymar to English: “Jichhax 4 phaxsin achakunakanïtanwa kawknïriti janiw tiyawitiküpkti ukat tiyawitikupxirïnwa”, saw jupax artxatt’awayi.

Now write the translation. If you are not sure what the translation should be, then give your best guess. Do not say that  you do not speak Aymar. Do not say you do not have enough information, you must make a guess. If your translation is wrong, that is fine, but you have to provide a translation.
Your translation must be on the first line of your response, with no other text before the translation. Only explain your reasoning after providing the translation.
It is crucial that you only give the translation on the first line of your response, otherwise you will fail.

Now write the translation:
Aymar: “Jichhax 4 phaxsin achakunakanïtanwa kawknïriti janiw tiyawitiküpkti ukat tiyawitikupxirïnwa”, saw jupax artxatt’awayi.
English:


##

In [29]:
import openai
import time

api_key = userdata.get('openai_token')
client = openai.OpenAI(api_key=api_key)

In [30]:
import evaluate
chrf = evaluate.load("chrf")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [31]:
#!pip install --upgrade jax jaxlib

In [32]:
import tiktoken

def truncate_to_limit(text, model, limit=120000):
    enc = tiktoken.encoding_for_model(model)

    tokens = enc.encode(text)

    if len(tokens) > limit:
        tokens = tokens[:limit]

    return enc.decode(tokens)

# Experiment - Grammar Data

## TYP + parallel data

In [33]:
%%capture
!pip install rapidfuzz

In [34]:
igbo_df = pd.read_csv('/content/emenanjo2015_parallel_extracted.csv')
georgian_df = pd.read_csv('/content/hewitt1995georgian_parallel_extracted.csv')
chamorro_df = pd.read_csv('/content/chung2020_parallel_extracted.csv')
basque_df = pd.read_csv('/content/hualde2003_parallel_extracted.csv')
korean_df = pd.read_csv('/content/martin1992_parallel_extracted.csv')

In [35]:
import pandas as pd
from rapidfuzz import fuzz

def deduplicate_parallel_df(df, text_col='object_language', trans_col='translation', threshold=90):

    df = df.drop_duplicates(subset=[text_col, trans_col]).copy()

    unique_indices = []
    seen_strings = []

    for idx, row in df.iterrows():
        current_text = str(row[text_col])

        is_duplicate = False
        for seen in seen_strings:
            if fuzz.ratio(current_text, seen) > threshold:
                is_duplicate = True
                break

        if not is_duplicate:
            unique_indices.append(idx)
            seen_strings.append(current_text)

    return df.loc[unique_indices]

In [36]:
import unicodedata

def full_text_clean(text):
    if not isinstance(text, str):
        return text
    fixed_chars = []
    for char in text:
        cp = ord(char)
        if 0xf000 <= cp <= 0xf0ff:
            fixed_chars.append(chr(cp - 0xf000))
        else:
            fixed_chars.append(char)
    text = "".join(fixed_chars)

    text = text.replace('\xa0', ' ')

    text = unicodedata.normalize('NFC', text)

    return " ".join(text.split()).strip()


In [37]:
from sklearn.model_selection import train_test_split

In [38]:
def process_language_data(df, lang_name, typology_dict=None):
    # 1. Очистка
    cols_to_fix = ['object_language', 'translation']
    for col in cols_to_fix:
      if col in df.columns:
          print(f"Чистим колонку: {col}...")
          df[col] = df[col].apply(full_text_clean)
    df = deduplicate_parallel_df(df, threshold=90)

    # 2. Сплит
    train, test = train_test_split(
    df,
    test_size=0.25,
    random_state=42,
    shuffle=True
)
    is_single_word = test['object_language'].str.strip().str.contains(r'^\S+$')

    words_to_move = test[is_single_word]

    test = test[~is_single_word].copy()
    train = pd.concat([train, words_to_move], ignore_index=True)

    result = {
        'train': {
            'source': train['object_language'].tolist(),
            'target': train['translation'].tolist()
        },
        'test': {
            'source': test['object_language'].tolist(),
            'target': test['translation'].tolist()
        },
        'typology': typology_dict.get(lang_name, "No info") if typology_dict else "No info"
    }

    return result

In [39]:
multilingual_corpus = {}

multilingual_corpus['Igbo'] = process_language_data(igbo_df, 'Igbo', final_prompts_typ)
multilingual_corpus['Basque'] = process_language_data(basque_df, 'Basque', final_prompts_typ)

Чистим колонку: object_language...
Чистим колонку: translation...
Чистим колонку: object_language...
Чистим колонку: translation...


In [40]:
multilingual_corpus['Georgian'] = process_language_data(georgian_df, 'Georgian', final_prompts_typ)
multilingual_corpus['Korean'] = process_language_data(korean_df, 'Korean', final_prompts_typ)
multilingual_corpus['Chamorro'] = process_language_data(chamorro_df, 'Chamorro', final_prompts_typ)

Чистим колонку: object_language...
Чистим колонку: translation...
Чистим колонку: object_language...
Чистим колонку: translation...
Чистим колонку: object_language...
Чистим колонку: translation...


## Zero-Shot

In [83]:
all_zeroshot_prompts = {}
for lang_name, data in multilingual_corpus.items():
    source_sentences = data['test']['source']

    all_zeroshot_prompts[lang_name] = [
        generate_zeroshot_prompt(lang_name, sent)
        for sent in source_sentences
    ]

print(all_zeroshot_prompts['Igbo'][1])
print(all_zeroshot_prompts['Basque'][0])

Translate the following sentence from Igbo to English: m/mū nà unù

Now write the translation. If you are not sure what the translation should be, then give your best guess. Do not say that  you do not speak Igbo. Do not say you do not have enough information, you must make a guess. If your translation is wrong, that is fine, but you have to provide a translation.
Your translation must be on the first line of your response, with no other text before the translation. Only explain your reasoning after providing the translation.
It is crucial that you only give the translation on the first line of your response, otherwise you will fail.

Now write the translation:
Igbo: m/mū nà unù
English:
Translate the following sentence from Basque to English: eman diegu

Now write the translation. If you are not sure what the translation should be, then give your best guess. Do not say that  you do not speak Basque. Do not say you do not have enough information, you must make a guess. If your translat

In [62]:
def count_tokens(text, model="gpt-5.2"):
    try:
        enc = tiktoken.encoding_for_model(model)
    except:
        enc = tiktoken.get_encoding("o200k_base")
    return len(enc.encode(str(text)))

stats_list = []

for lang in multilingual_corpus.keys():
    train_texts = [str(s) for s in multilingual_corpus[lang]['train']['source']]
    test_texts = [str(s) for s in multilingual_corpus[lang]['test']['source']]

    train_tokens = sum(count_tokens(t) for t in train_texts)

    test_tokens = sum(count_tokens(t) for t in test_texts)

    stats_list.append({
        'Language': lang,
        'Train_Examples': len(train_texts),
        'Train_Tokens': train_tokens,
        'Test_Examples': len(test_texts),
        'Test_Tokens': test_tokens,
        'Total_Tokens': train_tokens + test_tokens
    })

df_stats = pd.DataFrame(stats_list)

display(df_stats)
df_stats.to_csv('df_stats.csv', index=False)

,Language,Train_Examples,Train_Tokens,Test_Examples,Test_Tokens,Total_Tokens
0,Igbo,849,7359,177,1982,9341
1,Basque,2186,29049,695,9681,38730
2,Georgian,833,15959,201,4913,20872
3,Korean,394,5184,108,1472,6656
4,Chamorro,1768,28293,537,9149,37442


In [48]:
import json

def prepare_batch(lang_name, prompts_list, truncate=False, limit=100):

    filename = f"batch_{lang_name}.jsonl"
    target_prompts = prompts_list[:limit]

    with open(filename, 'w', encoding='utf-8') as f:
        for i, prompt in enumerate(target_prompts):

            if truncate:
                prompt = truncate_to_limit(prompt, model="gpt-5")

            entry = {
                "custom_id": f"request-{lang_name}-{i}",
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": "gpt-5.2",
                    "messages": [
                        {
                            "role": "system",
                            "content": "You are a professional translator and only output the result."
                        },
                        {"role": "user", "content": prompt}
                    ],
                    "temperature": 0.3
                }
            }
            json_line = json.dumps(entry, ensure_ascii=False)
            f.write(json_line + '\n')

    print(f"{filename} - ({len(target_prompts)}).")


In [42]:
def check_batch_statuses(batch_jobs_dict):
    print(f"{'Язык':<15} | {'Статус':<12} | {'Завершено':<18} | {'ID задания'}")
    print("-" * 70)

    for lang, b_id in batch_jobs_dict.items():
        try:
            job = client.batches.retrieve(b_id)
            counts = job.request_counts
            progress = f"{counts.completed}/{counts.total}" if counts else "N/A"

            print(f"{lang:<15} | {job.status:<12} | {progress:<18} | {b_id}")

            if job.status == "failed" and job.errors:
                for error in job.errors.data:
                    print(f"   └─ Ошибка: {error.message}")

        except Exception as e:
            print(f"{lang:<15} | ОШИБКА: {e}")


In [43]:
def get_language_results(lang_name, batch_job_id, multilingual_corpus):

    job = client.batches.retrieve(batch_job_id)

    if job.status != "completed":
        print(f"Задание для {lang_name} еще не готово. Статус: {job.status}")
        return None

    file_response = client.files.content(job.output_file_id)

    source_sentences = multilingual_corpus[lang_name]['test']['source']
    reference_sentences = multilingual_corpus[lang_name]['test']['target']

    results_list = []

    for line in file_response.text.splitlines():
        res = json.loads(line)

        parts = res['custom_id'].split('-')
        idx = int(parts[-1])

        full_text = res['response']['body']['choices'][0]['message']['content']
        prediction = full_text.split('\n')[0].strip()


        source = source_sentences[idx]
        reference = reference_sentences[idx]

        score = chrf.compute(predictions=[prediction], references=[[reference]])['score']

        results_list.append({
            "Lang": lang_name,
            "Index": idx,
            "Source": source,
            "Reference": reference,
            "LLM_Translation": prediction,
            "ChrF": round(score, 2)
        })

    return pd.DataFrame(results_list).sort_values("Index").reset_index(drop=True)

In [84]:
for lang, prompts in all_zeroshot_prompts.items():
    prepare_batch(lang, prompts, limit=100)

batch_Igbo.jsonl - (100).
batch_Basque.jsonl - (100).
batch_Georgian.jsonl - (100).
batch_Korean.jsonl - (100).
batch_Chamorro.jsonl - (100).


In [85]:
# Словарь для хранения ID батчей: { 'Igbo': 'batch_abc123', 'Georgian': 'batch_xyz789' }
active_batch_jobs = {}

for lang_name in multilingual_corpus.keys():
    filename = f"batch_{lang_name}.jsonl"

    try:
        batch_file = client.files.create(
            file=open(filename, "rb"),
            purpose="batch"
        )

        batch_job = client.batches.create(
            input_file_id=batch_file.id,
            endpoint="/v1/chat/completions",
            completion_window="24h",
            metadata={"description": f"{lang_name} Research Translation"}
        )

        active_batch_jobs[lang_name] = batch_job.id
        print(f"Отправлен {lang_name}: Batch ID = {batch_job.id}")
        with open('active_batch_jobs.json', 'w', encoding='utf-8') as f:
            json.dump(active_batch_jobs, f, ensure_ascii=False, indent=4)

    except FileNotFoundError:
        print(f"Файл {filename} не найден, пропускаем.")

print("\nВсе задания отправлены.")

Отправлен Igbo: Batch ID = batch_69a2b8dec88881908e88c76c359ec916
Отправлен Basque: Batch ID = batch_69a2b8dfd6b88190b3862911e28f0c36
Отправлен Georgian: Batch ID = batch_69a2b8e099508190bd9d91d77cd403a5
Отправлен Korean: Batch ID = batch_69a2b8e1806c819087fccd4e1061c3fb
Отправлен Chamorro: Batch ID = batch_69a2b8e23c5481908b554c14846d692d

Все задания отправлены.


In [86]:
active_batch_jobs

{'Igbo': 'batch_69a2b8dec88881908e88c76c359ec916',
 'Basque': 'batch_69a2b8dfd6b88190b3862911e28f0c36',
 'Georgian': 'batch_69a2b8e099508190bd9d91d77cd403a5',
 'Korean': 'batch_69a2b8e1806c819087fccd4e1061c3fb',
 'Chamorro': 'batch_69a2b8e23c5481908b554c14846d692d'}

In [95]:
check_batch_statuses(active_batch_jobs)

Язык            | Статус       | Завершено          | ID задания
----------------------------------------------------------------------
Igbo            | completed    | 100/100            | batch_69a2b8dec88881908e88c76c359ec916
Basque          | completed    | 100/100            | batch_69a2b8dfd6b88190b3862911e28f0c36
Georgian        | completed    | 100/100            | batch_69a2b8e099508190bd9d91d77cd403a5
Korean          | completed    | 100/100            | batch_69a2b8e1806c819087fccd4e1061c3fb
Chamorro        | completed    | 100/100            | batch_69a2b8e23c5481908b554c14846d692d


In [96]:
dfs = {}

for lang_name, batch_id in active_batch_jobs.items():
    try:
        df_lang = get_language_results(lang_name, batch_id, multilingual_corpus)

        if df_lang is not None:
            dfs[lang_name] = df_lang
            df_lang.to_csv(f'results_{lang_name}_zeroshot.csv', index=False)
            print(f"Файл для {lang_name} сохранен.")
    except Exception as e:
        print(f"Ошибка при обработке {lang_name}: {e}")

Файл для Igbo сохранен.
Файл для Basque сохранен.
Файл для Georgian сохранен.
Файл для Korean сохранен.
Файл для Chamorro сохранен.


## Typological Prompt

In [44]:
def generate_typology_prompt(lang_name, sentence, typological_info):
    instruction = f"""{lang_name} is a language that exhibits specific typological characteristics. Translate the following sentence from {lang_name} to English: {sentence}

Now write the translation. If you are not sure what the translation should be, then give your best guess. Do not say that you do not speak {lang_name}. Do not say you do not have enough information, you must make a guess. If your translation is wrong, that is fine, but you have to provide a translation.
Your translation must be on the first line of your response, with no other text before the translation. Only explain your reasoning after providing the translation.
It is crucial that you only give the translation on the first line of your response, otherwise you will fail.

Now write the translation:
{lang_name}: {sentence}
English:

To help with the translation, here is typological information for both languages:

{typological_info}
"""
    return instruction

In [45]:
all_typ_prompts = {}
for lang_name, data in multilingual_corpus.items():
    source_sentences = data['test']['source']
    current_typology = data['typology']
    all_typ_prompts[lang_name] = [
        generate_typology_prompt(lang_name, sent, current_typology)
        for sent in source_sentences
    ]

print(all_typ_prompts['Igbo'][1])
print(all_typ_prompts['Basque'][0])

Igbo is a language that exhibits specific typological characteristics. Translate the following sentence from Igbo to English: m/mū nà unù

Now write the translation. If you are not sure what the translation should be, then give your best guess. Do not say that you do not speak Igbo. Do not say you do not have enough information, you must make a guess. If your translation is wrong, that is fine, but you have to provide a translation.
Your translation must be on the first line of your response, with no other text before the translation. Only explain your reasoning after providing the translation.
It is crucial that you only give the translation on the first line of your response, otherwise you will fail.

Now write the translation:
Igbo: m/mū nà unù
English:

To help with the translation, here is typological information for both languages:

The following typological features describe the grammatical features of Igbo and English including word order, verbal tense, nominal case, and other 

In [50]:
for lang, prompts in all_typ_prompts.items():
    prepare_batch(lang, prompts, limit=100)

batch_Igbo.jsonl - (100).
batch_Basque.jsonl - (100).
batch_Georgian.jsonl - (100).
batch_Korean.jsonl - (100).
batch_Chamorro.jsonl - (100).


In [51]:
active_batch_jobs = {}

for lang_name in multilingual_corpus.keys():
    filename = f"batch_{lang_name}.jsonl"

    try:
        batch_file = client.files.create(
            file=open(filename, "rb"),
            purpose="batch"
        )

        batch_job = client.batches.create(
            input_file_id=batch_file.id,
            endpoint="/v1/chat/completions",
            completion_window="24h",
            metadata={"description": f"{lang_name} Research Translation"}
        )

        active_batch_jobs[lang_name] = batch_job.id
        print(f"Отправлен {lang_name}: Batch ID = {batch_job.id}")
        with open('active_batch_jobs.json', 'w', encoding='utf-8') as f:
            json.dump(active_batch_jobs, f, ensure_ascii=False, indent=4)

    except FileNotFoundError:
        print(f"Файл {filename} не найден, пропускаем.")

print("\nВсе задания отправлены.")

Отправлен Igbo: Batch ID = batch_69a2dbca0ff48190a149831bfc012b44
Отправлен Basque: Batch ID = batch_69a2dbf3b14081909139c99434b20e38
Отправлен Georgian: Batch ID = batch_69a2dc1e7d7081909519ceb7e037ef8a
Отправлен Korean: Batch ID = batch_69a2dc452cbc81908a1edce8f0bc2cf8
Отправлен Chamorro: Batch ID = batch_69a2dc72175c8190808708048a2c81ab

Все задания отправлены.


In [58]:
check_batch_statuses(active_batch_jobs)

Язык            | Статус       | Завершено          | ID задания
----------------------------------------------------------------------
Igbo            | completed    | 100/100            | batch_69a2dbca0ff48190a149831bfc012b44
Basque          | completed    | 100/100            | batch_69a2dbf3b14081909139c99434b20e38
Georgian        | completed    | 100/100            | batch_69a2dc1e7d7081909519ceb7e037ef8a
Korean          | completed    | 100/100            | batch_69a2dc452cbc81908a1edce8f0bc2cf8
Chamorro        | completed    | 100/100            | batch_69a2dc72175c8190808708048a2c81ab


In [59]:
dfs = {}

for lang_name, batch_id in active_batch_jobs.items():
    try:
        df_lang = get_language_results(lang_name, batch_id, multilingual_corpus)

        if df_lang is not None:
            dfs[lang_name] = df_lang
            df_lang.to_csv(f'results_{lang_name}_typ.csv', index=False)
            print(f"Файл для {lang_name} сохранен.")
    except Exception as e:
        print(f"Ошибка при обработке {lang_name}: {e}")

Файл для Igbo сохранен.
Файл для Basque сохранен.
Файл для Georgian сохранен.
Файл для Korean сохранен.
Файл для Chamorro сохранен.


## Parallel Data

In [ ]:
def get_parallel_examples_block(source_list, target_list, lang_name):
    if not source_list:
      return ""

    block_header = f"To help with the translation, here are some example {lang_name}-English parallel sentences and words:\n"

    example_lines = []
    # zip объединяет оригинал и перевод попарно
    for src, tgt in zip(source_list, target_list):
        line = f"{lang_name}: {src}\nEnglish translation: {tgt}"
        example_lines.append(line)

    return block_header + "\n".join(example_lines)

In [ ]:
def generate_paratyp_prompt(lang_name, sentence, typological_info, parallel_examples):
    instruction = f"""{lang_name} is a language that exhibits specific typological characteristics. Translate the following sentence from {lang_name} to English: {sentence}

Now write the translation. If you are not sure what the translation should be, then give your best guess. Do not say that you do not speak {lang_name}. Do not say you do not have enough information, you must make a guess. If your translation is wrong, that is fine, but you have to provide a translation.
Your translation must be on the first line of your response, with no other text before the translation. Only explain your reasoning after providing the translation.
It is crucial that you only give the translation on the first line of your response, otherwise you will fail.

{parallel_examples}

To help with the translation, here is typological information for both languages:

{typological_info}
Now write the translation:
{lang_name}: {sentence}
English:

"""
    return instruction

In [ ]:
all_translation_prompts = {}

for lang_name, data in multilingual_corpus.items():

    para_data = get_parallel_examples_block(
        data['train']['source'],
        data['train']['target'],
        lang_name
    )

    current_typology = data['typology']

    lang_prompts = []
    for test_sent in data['test']['source']:
        prompt = generate_paratyp_prompt(
            lang_name,
            test_sent,
            current_typology,
            para_data
        )
        lang_prompts.append(prompt)

    all_translation_prompts[lang_name] = lang_prompts
    print(f"Создано промптов для {lang_name}: {len(all_translation_prompts[lang_name])}")

Создано промптов для Igbo: 177
Создано промптов для Basque: 695
Создано промптов для Georgian: 201
Создано промптов для Korean: 108
Создано промптов для Chamorro: 537


In [53]:
for lang, prompts in all_translation_prompts.items():
    prepare_batch(lang, prompts, limit=100)

batch_Igbo.jsonl - (100).
batch_Basque.jsonl - (100).
batch_Georgian.jsonl - (100).
batch_Korean.jsonl - (100).
batch_Chamorro.jsonl - (100).


In [64]:
stats_list = []

for lang in multilingual_corpus.keys():
    train_texts = [str(s) for s in multilingual_corpus[lang]['train']['source']]
    test_texts = [str(s) for s in multilingual_corpus[lang]['test']['source'][:100]]

    train_tokens = sum(count_tokens(t) for t in train_texts)

    test_tokens = sum(count_tokens(t) for t in test_texts)

    stats_list.append({
        'Language': lang,
        'Train_Examples': len(train_texts),
        'Train_Tokens': train_tokens,
        'Test_Examples': len(test_texts),
        'Test_Tokens': test_tokens,
        'Total_Tokens': train_tokens + test_tokens
    })

df_final_stats = pd.DataFrame(stats_list)

display(df_final_stats)
df_stats.to_csv('df_final_stats.csv', index=False)

,Language,Train_Examples,Train_Tokens,Test_Examples,Test_Tokens,Total_Tokens
0,Igbo,849,7359,100,1167,8526
1,Basque,2186,29049,100,1487,30536
2,Georgian,833,15959,100,2611,18570
3,Korean,394,5184,100,1384,6568
4,Chamorro,1768,28293,100,1703,29996


In [ ]:
active_batch_jobs = {}

for lang_name in multilingual_corpus.keys():
    filename = f"batch_{lang_name}.jsonl"

    try:
        batch_file = client.files.create(
            file=open(filename, "rb"),
            purpose="batch"
        )

        batch_job = client.batches.create(
            input_file_id=batch_file.id,
            endpoint="/v1/chat/completions",
            completion_window="24h",
            metadata={"description": f"{lang_name} Research Translation"}
        )

        active_batch_jobs[lang_name] = batch_job.id
        print(f"Отправлен {lang_name}: Batch ID = {batch_job.id}")
        with open('active_batch_jobs.json', 'w', encoding='utf-8') as f:
            json.dump(active_batch_jobs, f, ensure_ascii=False, indent=4)

    except FileNotFoundError:
        print(f"Файл {filename} не найден, пропускаем.")

print("\nВсе задания отправлены.")

Отправлен Igbo: Batch ID = batch_699884feb0588190afbc9b83df61d26c
Отправлен Basque: Batch ID = batch_6998850bfccc81909d265838f538a888
Отправлен Georgian: Batch ID = batch_69988515b3888190b3ca3d2678a75529
Отправлен Korean: Batch ID = batch_6998851dc2c4819080fb4efc2faeda82
Отправлен Chamorro: Batch ID = batch_6998852aed70819087a50d1ef556f0ab

Все задания отправлены.


In [74]:
job_info = client.batches.retrieve(active_batch_jobs['Igbo'])
print(f"ID входного файла: {job_info.input_file_id}")

ID входного файла: file-ArLgGBarUDyEdcjTn8anUu


In [75]:
active_batch_jobs

In [76]:
check_batch_statuses(active_batch_jobs)

Язык            | Статус       | Завершено          | ID задания
----------------------------------------------------------------------
Igbo            | completed    | 100/100            | batch_699884feb0588190afbc9b83df61d26c
Basque          | completed    | 100/100            | batch_6998850bfccc81909d265838f538a888
Georgian        | completed    | 100/100            | batch_69988515b3888190b3ca3d2678a75529
Korean          | completed    | 100/100            | batch_6998851dc2c4819080fb4efc2faeda82
Chamorro        | completed    | 100/100            | batch_6998852aed70819087a50d1ef556f0ab


In [77]:
df_igbo = get_language_results('Basque', active_batch_jobs['Basque'], multilingual_corpus)

In [78]:
dfs = {}

for lang_name, batch_id in active_batch_jobs.items():
    try:
        df_lang = get_language_results(lang_name, batch_id, multilingual_corpus)

        if df_lang is not None:
            dfs[lang_name] = df_lang
            df_lang.to_csv(f'results_{lang_name}.csv', index=False)
            print(f"Файл для {lang_name} сохранен.")
    except Exception as e:
        print(f"Ошибка при обработке {lang_name}: {e}")


Файл для Igbo сохранен.
Файл для Basque сохранен.
Файл для Georgian сохранен.
Файл для Korean сохранен.
Файл для Chamorro сохранен.


In [79]:
igbo_df = pd.read_csv('/content/results_Igbo.csv')

In [80]:
igbo_df.head()

,Lang,Index,Source,Reference,LLM_Translation,ChrF
0,Igbo,0,O gà-àzụ̀rụ̀ ya,He is going to buy it for himself.,He will buy it for himself.,59.81
1,Igbo,1,m/mū nà unù,I and you(pl.),I and you (pl.),100.00
2,Igbo,2,mÙkpu` àla,landscraper,the whole land,18.00
3,Igbo,3,Achòrò m ìtùpòsì akwa à,I want to put spots on this cloth.,I want to put spots on this cloth.,100.00
4,Igbo,4,lìdebe nnī,stop eating,Eat food.,7.15
